# Syqnal Glasses: train the vision models on real images (free GPU, about 1 hour)

Works on **Google Colab** and **Kaggle**. To finish in about an hour, run it twice at the same time:
Colab with `TRAIN = 'products'` and Kaggle with `TRAIN = 'tags'`. Or set `TRAIN = 'both'` on one machine (about 1.5 to 2 hours).

| Model | Real data | Licence |
| --- | --- | --- |
| Shelf-product detector | SKU-110K, real supermarket shelves, every pack boxed | Non-commercial: prototype only, replace before launch |
| Price-tag detector | Open Food Facts `price-tag-detection`, 2,229 shelf photos | CC BY-SA 4.0 |

**Setup:**
- **Colab:** Runtime, then Change runtime type, then **T4 GPU**, then Run all. Models are saved to Google Drive: `MyDrive/syqnal_runs`.
- **Kaggle:** Settings, then Accelerator **GPU T4**, Internet **On** (needs a verified phone number), then Run all. Models are saved under Output (`/kaggle/working/syqnal_runs`); download them from the Output tab.

Timings are estimates for a free T4, not measured.

In [ ]:
TRAIN = 'products'      # 'products' (Colab), 'tags' (Kaggle), or 'both'
N_SKU_SHARDS = 5        # SKU-110K training shards to use, about 430 photos each
EPOCHS_PRODUCTS = 8
EPOCHS_TAGS = 12

In [ ]:
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv
if os.path.exists('/kaggle'):
    OUT = '/kaggle/working/syqnal_runs'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/syqnal_runs'
os.makedirs(OUT, exist_ok=True); print('Models will be saved in', OUT)

In [ ]:
WORK = '/kaggle/working/syqnalglasses' if os.path.exists('/kaggle') else '/content/syqnalglasses'
!git clone -q -b vision-pipeline https://github.com/cheunglokkwan-design/syqnalglasses.git {WORK}
%cd {WORK}
!pip -q install "rfdetr[train]>=1.12" "transformers>=4.40" rapidocr-onnxruntime zxing-cpp

In [ ]:
B = 'https://huggingface.co/datasets'
!mkdir -p data/real/raw/pricetags data/real/raw/sku110k
if TRAIN in ('tags', 'both'):
    !wget -q -O data/real/raw/pricetags/train.parquet {B}/openfoodfacts/price-tag-detection/resolve/main/data/train-00000-of-00001.parquet
    !wget -q -O data/real/raw/pricetags/val.parquet {B}/openfoodfacts/price-tag-detection/resolve/main/data/val-00000-of-00001.parquet
    !python vision/tools/real_to_coco.py --raw data/real/raw --out data/real --only tags
if TRAIN in ('products', 'both'):
    for i in range(N_SKU_SHARDS):
        n = f'{i:05d}'
        !wget -q -O data/real/raw/sku110k/train-{n}.parquet {B}/benjamintli/sku110k/resolve/main/data/train-{n}-of-00019.parquet
    !wget -q -O data/real/raw/sku110k/validation-00000.parquet {B}/benjamintli/sku110k/resolve/main/data/validation-00000-of-00002.parquet
    !python vision/tools/real_to_coco.py --raw data/real/raw --out data/real --only products

## Train

In [ ]:
if TRAIN in ('products', 'both'):
    !python vision/tools/train_detector.py --data data/real/coco_products --out {OUT}/det_products --size nano --resolution 640 --epochs {EPOCHS_PRODUCTS} --batch 8 --accum 2 --workers 2
if TRAIN in ('tags', 'both'):
    !python vision/tools/train_detector.py --data data/real/coco_tags --out {OUT}/det_tags --size nano --resolution 640 --epochs {EPOCHS_TAGS} --batch 8 --accum 2 --workers 2
!ls -la {OUT}/*

## Check on real test photos

In [ ]:
import glob, sys
from PIL import Image, ImageDraw
from IPython.display import display
sys.path.insert(0, 'vision')
from aislelens_vision.detector import load_detectors
det = load_detectors([f'{OUT}/det_products/checkpoint_best_total.pth', f'{OUT}/det_tags/checkpoint_best_total.pth'])
tests = sorted(glob.glob('data/real/coco_tags/test/*.jpg'))[:3] + sorted(glob.glob('data/real/coco_products/test/*.jpg'))[:2]
for f in tests:
    im = Image.open(f).convert('RGB'); d = ImageDraw.Draw(im)
    for b in det(im):
        d.rectangle([b['x0'], b['y0'], b['x1'], b['y1']], outline='yellow' if b['label'] == 'price_tag' else 'lime', width=3)
    im.thumbnail((900, 900)); display(im)

## Then

Keep `checkpoint_best_total.pth` and `detector.json` from `det_products/` and `det_tags/`. On the laptop they go in the repo's `runs/det_products/` and `runs/det_tags/`; `python vision/serve.py` then uses them automatically.